In [ ]:
import random
import numpy as np
from copy import deepcopy

import gymnasium as gym
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import ResizeObservation, GrayscaleObservation, FrameStackObservation, TransformObservation
import numpy as np

def create_env(mode="rgb_array"):
    env = gym.make(
        "CarRacing-v3", 
        render_mode=mode, 
        lap_complete_percent=0.95, 
        domain_randomize=False, 
        continuous=False,
    )

    env = GrayscaleObservation(env, keep_dim=False)

    env = ResizeObservation(env, (84, 84))

    env = FrameStackObservation(env, stack_size=4)

    final_space = gym.spaces.Box(
        low=0, high=1, shape=(4, 84, 84), dtype=np.float32
    )
    
    env = TransformObservation(
        env, 
        lambda obs: (np.array(obs) / 255.0).astype(np.float32),
        observation_space=final_space
    )

    return env

In [ ]:
env = create_env()

In [ ]:


class ReplayBuffer:

    def __init__(self, capacity=10_000, batch_size=32):
        self.capacity = capacity
        self.batch_size = batch_size
        self.replay_buffer = deque(maxlen=capacity)

    def can_sample(self):
        return (self.batch_size * 4) <= len(self.replay_buffer)

    def store_transition(self, state, action, reward, next_state, done):
        self.replay_buffer.append([
            state, action, reward, next_state, done
        ])

    def sample(self):
        sampled_data = random.sample(self.replay_buffer, self.batch_size)
        states, actions, rewards, next_states, dones = list(zip(*sampled_data))
        states = np.stack(states)
        actions = np.stack(actions)
        rewards = np.stack(rewards)
        next_states = np.stack(next_states)
        dones = np.stack(dones)

        return states, actions, rewards, next_states, dones

In [ ]:
# replay_buffer = ReplayBuffer()

# for _ in range(100):
#     done = False
#     state, _ = env.reset()
#     while not done:
#         action = env.action_space.sample()
#         next_state, reward, terminated, truncated, _ = env.step(action)
#         done = terminated or truncated
#         replay_buffer.store_transition(
#             state, action, reward, next_state, done
#         )
#         state = next_state


In [ ]:
# replay_buffer.sample()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class QNetwork(nn.Module):

    def __init__(self, in_features=4, hidden_size=512, n_actions=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_features, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
        )

        # 64 * 7 * 7 = 3136
        self.advantage = nn.Sequential(
            nn.Linear(3136, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions),
        )

        self.value = nn.Sequential(
            nn.Linear(3136, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        features = self.conv(x).reshape(x.size(0), -1)
        advantage = self.advantage(features)
        value = self.value(features)
        return value + (advantage - advantage.mean(dim=1, keepdim=True))


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
class DQN:

    def __init__(self, env, hidden_size, buffer_size, alpha=0.1, gamma=0.99, batch_size=64, lr=1e-4, episodes=3000, epsilon=1.0, min_epsilon=0.1, decay_rate=1e-5):
        self.env = env
        self.hidden_size = hidden_size
        self.buffer_size = buffer_size
        self.batch_size = batch_size
        self.alpha = alpha
        self.gamma = gamma
        self.n = 3
        self.episodes = episodes
        self.epsilon = epsilon
        self.min_epsilon = min_epsilon
        self.decay_rate = decay_rate
        self.steps = 0
        self.average_q_value = 0.

        self.state_size = 4
        self.n_actions = env.action_space.n
        print(f"State size: {self.state_size}, Number of actions: {self.n_actions}")

        self.q_network = QNetwork(in_features=self.state_size, hidden_size=hidden_size, n_actions=self.n_actions).to(device)
        self.q_network.load_state_dict(torch.load("car_racing.pth"))
        self.target_network = deepcopy(self.q_network).eval()

        self.replay_buffer = ReplayBuffer(buffer_size, batch_size)

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)

    def state_to_tensor(self, state):
        state = torch.from_numpy(state).unsqueeze(0).to(device)
        return state
    
    def batch_to_tensor(self, batch):
        states, actions, rewards, next_states, dones = batch
        states = torch.from_numpy(states).float().to(device)
        actions = torch.from_numpy(actions).unsqueeze(1).to(device)
        rewards = torch.from_numpy(rewards).float().unsqueeze(1).to(device)
        next_states = torch.from_numpy(next_states).float().to(device)
        dones = torch.from_numpy(dones).bool().unsqueeze(1).to(device)

        return states, actions, rewards, next_states, dones
    
    def policy(self, state, epsilon=None):
        if epsilon is None:
            epsilon = self.epsilon
        if random.random() < epsilon:
            return self.env.action_space.sample()
        with torch.no_grad():
            q_values = self.q_network(self.state_to_tensor(state))
            action = torch.argmax(q_values).cpu().item()
            return action
        
    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon - self.decay_rate)

    def update_target_q_network(self):
        self.target_network.load_state_dict(self.q_network.state_dict())

    def step(self):
        states, actions, rewards, next_states, dones = self.replay_buffer.sample()
        states, actions, rewards, next_states, dones = self.batch_to_tensor((states, actions, rewards, next_states, dones))
        q_values = self.q_network(states)
        selected_q_values = q_values.gather(1, actions)
        with torch.no_grad():
            target_main_q_values = self.q_network(next_states)
            _, next_actions = target_main_q_values.max(dim=1, keepdim=True)
            target_q_values: torch.Tensor = self.target_network(next_states)
            selected_target_q_values = target_q_values.gather(1, next_actions)
            targets =  rewards + (self.gamma ** self.n) * ~dones * selected_target_q_values
            self.average_q_value = 0.99 * self.average_q_value + 0.01 * selected_q_values.mean().item()

        self.optimizer.zero_grad()
        loss = F.smooth_l1_loss(selected_q_values, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)
        self.optimizer.step()

    def train_on_episode(self, episode=None):
        n_step_buffer = deque(maxlen=self.n)
        done = False
        state, _ = self.env.reset()
        episode_reward = 0.
        while not done:
            action = self.policy(state)
            
            next_state, reward, terminated, truncated, _ = self.env.step(action)
            
            done = terminated or truncated
            
            n_step_buffer.append((state, action, reward))

            if len(n_step_buffer) == self.n:
                G = sum([(self.gamma ** i) * n_step_buffer[i][2] for i in range(self.n)])
                
                state_tau, action_tau, _ = n_step_buffer[0]

                self.replay_buffer.store_transition(
                    state_tau, action_tau, G, next_state, done
                )

            state = next_state
            
            if self.replay_buffer.can_sample():
                self.step()
                self.decay_epsilon()

                if self.steps % 5_000 == 0:
                    self.update_target_q_network()
                
            episode_reward += reward
            self.steps += 1

        while len(n_step_buffer) > 0:
            G = sum([(self.gamma ** i) * n_step_buffer[i][2] for i in range(len(n_step_buffer))])
            
            state_tau, action_tau, _ = n_step_buffer.popleft()
            
            self.replay_buffer.store_transition(
                state_tau, action_tau, G, next_state, True
            )

        return episode_reward

    def train(self):
        self.q_network.train()
        episode_rewards = deque(maxlen=100)
        for episode in range(1, self.episodes + 1):
            episode_reward = self.train_on_episode(episode)
            episode_rewards.append(episode_reward)

            if episode % 10 == 0:
                print(f"Episode: {episode}, Steps: {self.steps},  Reward: {np.mean(episode_rewards)}, Average Q-Value: {self.average_q_value}, Epsilon: {self.epsilon}")
        


    def test(self, episode=10):
        self.q_network.eval()
        env = create_env()
        with torch.no_grad():
            for _ in range(episode):
                done = False
                state, _ = env.reset()
                episode_reward = 0.
                while not done:
                    action = self.policy(state, epsilon=0.)
                    next_state, reward, terminated, truncated, _ = env.step(action)
                    done = terminated or truncated
                    state = next_state
                    episode_reward += reward

                print(f"Reward: {episode_reward}")
        env.close()
            

        


In [ ]:
dqn = DQN(env, hidden_size=512, buffer_size=100_000, lr=1e-4, batch_size=32, episodes=100_000, epsilon=1, min_epsilon=0.05, decay_rate=1e-6)

In [ ]:
# dqn.train()


In [ ]:
dqn.test(3)

In [ ]:
def test(episode=1):
    dqn.q_network.eval()
    env = create_env("human")
    with torch.no_grad():
        for _ in range(episode):
            done = False
            state, _ = env.reset()
            episode_reward = 0.
            while not done:
                action = dqn.policy(state, epsilon=0.0)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated
                state = next_state
                episode_reward += reward

            print(f"Reward: {episode_reward}")
    env.close()

test()

In [ ]:
torch.save(dqn.q_network.state_dict(), "car_racing.pth")